# 实验性车辆形式抓取笔记本

用于测试数据状态以及生成形式数据

## Part.1 类型抓取

In [1]:
import httpx
from bs4 import BeautifulSoup
import re
import json
import pandas as pd
import os
current_dir = os.getcwd()
PROJECT_ROOT = os.path.dirname(os.path.dirname(current_dir))
PROJECT_ROOT

'/Users/yukun/projects/wakareeru'

In [2]:
def fetch_wikitext(page_title: str) -> str:
    """获取页面的原始Wikitext"""
    url = "https://ja.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "titles": page_title,
        "prop": "revisions",
        "rvprop": "content",     # 返回原始wikitext
        "rvslots": "main",
        "format": "json",
    }
    resp = httpx.get(url, params=params)
    pages = resp.json()["query"]["pages"]
    page = next(iter(pages.values()))
    return page["revisions"][0]["slots"]["main"]["*"]

In [3]:
import asyncio

OPERATORS = [
    ("JR東日本", "JR East",    "JR東日本の車両形式"),
   #("JR東海",   "JR Central", "JR東海の車両形式"),
    #("JR西日本", "JR West",    "JR西日本の車両形式"),
    ("JR貨物", "JR Freight", "JR貨物の車両形式"),
]

HEADERS = {
    "User-Agent": "JapaneseTrainDatasetBuilder/1.0 (research project; fengyukunfyk@gmail.com)"
}

async def _fetch_one(
    client: httpx.AsyncClient, operator_jp: str, operator_en: str, page_title: str
) -> tuple[str, str, str, str]:
    params = {
        "action": "query",
        "titles": page_title,
        "prop": "revisions",
        "rvprop": "content",
        "rvslots": "main",
        "format": "json",
    }
    resp = await client.get("https://ja.wikipedia.org/w/api.php", params=params)
    resp.raise_for_status()
    pages = resp.json()["query"]["pages"]
    page = next(iter(pages.values()))
    print(f"页面：{page_title} 请求成功")
    return page_title, operator_jp, operator_en, page["revisions"][0]["slots"]["main"]["*"]

async def fetch_all(operators: list[tuple[str, str, str]]) -> dict[str, tuple[str, str, str]]:
    async with httpx.AsyncClient(headers=HEADERS, timeout=30) as client:
        results = await asyncio.gather(*[_fetch_one(client, jp, en, page) for jp, en, page in operators])
    # {page_title: (operator_jp, operator_en, wikitext)}
    return {page: (jp, en, wt) for page, jp, en, wt in results}

wikitexts = await fetch_all(OPERATORS)
wikitexts["JR東日本の車両形式"][2][:500]  # 预览

页面：JR貨物の車両形式 请求成功
页面：JR東日本の車両形式 请求成功


"{{Pathnav|[[東日本旅客鉄道|東日本旅客鉄道（JR東日本）]]|frame=1}}\n'''JR東日本の車両形式'''（ジェイアールひがしにほんのしゃりょうけいしき）は、[[東日本旅客鉄道]]（JR東日本）に在籍する、あるいは在籍した[[鉄道車両]]の一覧である。\n\n== 現在の所属車両 ==\n=== 新幹線電車 ===\n*'''営業用'''\n**[[新幹線E2系電車|E2系]]\n**[[新幹線E3系電車|E3系]]\n**[[新幹線E5系・H5系電車|E5系]]\n**[[新幹線E6系電車|E6系]]\n**[[新幹線E7系・W7系電車|E7系]]\n**[[新幹線E8系電車|E8系]]\n*'''事業用'''\n**[[新幹線E926形電車|E926形]]（電気・軌道総合検測車）\n**[[新幹線E956形電車|E956形]]（高速試験車）\n\n=== 蒸気機関車 ===\n* [[国鉄C57形蒸気機関車|C57形]] - [[国鉄C57形蒸気機関車180号機|180号機]]\n* [[国鉄C58形蒸気機関車|C58形]] - [[国鉄C58形蒸気機関車#動態保存機|239号機]]{{efn2"

In [10]:
import re

STATUS_MAP = {
    "現在の所属車両":   "現役",
    "過去の所属車両":   "廃止",
    "在来線現有車両":   "現役",
    "在来線廃止車両":   "廃止",
    "導入予定車両":     "導入予定",
}

def parse_vehicle_wikitext(lines: list[str]) -> list[dict]:
    link_re = re.compile(r'\[\[([^\]|]+)(?:\|([^\]]+))?\]\]')

    # 首字符允许数字，覆盖 113系/271系 等纯数字开头的 JR West/Central 型号
    series_re = re.compile(
        r'^[A-Za-z゠-ヿ一-鿿\d]'  # 首字符（含数字）
        r'[A-Za-z゠-ヿ一-鿿\-]*'   # 前缀（含连字符，如HB-E）
        r'\d+(?:系|形)?$'           # 数字结尾，可选系/形
    )

    results = []
    current_h2 = ""
    current_h3 = ""
    current_subtype = ""
    skip_sections = {"2010年4月1日時点の在籍貨車", "2010年度以降に新製が発表された貨車", "2009年度までに消滅した貨車", # JR货物用的排除项
                     "脚注", "注釈", "出典", "関連項目", "外部リンク"}

    for line in lines:
        line = line.rstrip('\n')

        m = re.match(r'^== (.+?) ==$', line)
        if m:
            current_h2 = m.group(1)
            current_h3 = ""
            current_subtype = ""
            continue

        m = re.match(r'^=== (.+?) ===$', line)
        if m:
            current_h3 = m.group(1)
            current_subtype = ""
            continue

        if current_h2 in skip_sections:
            continue

        if not line.lstrip().startswith('*'):
            continue

        # 单星开头的粗体行（* '''xxx'''）才更新 subtype，** 及以上层级不更新
        subtype = re.match(r'^\*\s*\'\'\'(.+?)\'\'\'', line)
        if subtype:
            current_subtype = subtype.group(1).strip().strip('[]')

        for m in link_re.finditer(line):
            page  = m.group(1)
            label = m.group(2) or page
            label = label.split('・')[0].split('（')[0].strip()

            if series_re.match(label):
                results.append({
                    "series":     label,
                    "wiki_title": page,
                    "status":     STATUS_MAP.get(current_h2, current_h2),
                    "type":       current_h3,
                    "subtype":    current_subtype,
                })

    return results

In [11]:
for page_title, (operator_jp, operator_en, wt) in wikitexts.items():
    print(f"=== {operator_en} ({operator_jp}) — {page_title} ===")
    print(wt[:500])
    print()

=== JR East (JR東日本) — JR東日本の車両形式 ===
{{Pathnav|[[東日本旅客鉄道|東日本旅客鉄道（JR東日本）]]|frame=1}}
'''JR東日本の車両形式'''（ジェイアールひがしにほんのしゃりょうけいしき）は、[[東日本旅客鉄道]]（JR東日本）に在籍する、あるいは在籍した[[鉄道車両]]の一覧である。

== 現在の所属車両 ==
=== 新幹線電車 ===
*'''営業用'''
**[[新幹線E2系電車|E2系]]
**[[新幹線E3系電車|E3系]]
**[[新幹線E5系・H5系電車|E5系]]
**[[新幹線E6系電車|E6系]]
**[[新幹線E7系・W7系電車|E7系]]
**[[新幹線E8系電車|E8系]]
*'''事業用'''
**[[新幹線E926形電車|E926形]]（電気・軌道総合検測車）
**[[新幹線E956形電車|E956形]]（高速試験車）

=== 蒸気機関車 ===
* [[国鉄C57形蒸気機関車|C57形]] - [[国鉄C57形蒸気機関車180号機|180号機]]
* [[国鉄C58形蒸気機関車|C58形]] - [[国鉄C58形蒸気機関車#動態保存機|239号機]]{{efn2

=== JR Freight (JR貨物) — JR貨物の車両形式 ===
{{Pathnav|[[日本貨物鉄道|日本貨物鉄道（JR貨物）]]|frame=1}}
'''JR貨物の車両形式'''（ジェイアールかもつのしゃりょうけいしき）は、[[日本貨物鉄道]]（JR貨物）に在籍する、あるいは在籍した[[鉄道車両]]の一覧である。

JR貨物に所属する車両の新製・[[廃車 (鉄道)|廃車]]・改造などについては[[国鉄分割民営化]]以降2009年度までは他の鉄道事業者と同様に鉄道雑誌・[[年鑑]]等へ前年度の情報を提供し公表していたが、2010年度以降、鉄道雑誌・年鑑へは一部を除き所属車両の動態についての公表や情報提供を行っていない{{efn|一例として、[[交友社]]『[[鉄道ファン (雑誌)|鉄道ファン]]』において毎年7月号で行っている特集「JR車両ファイル」、[[交通新聞社]] ジェー・アール・アール『JR気動車・客車編成表』、[[電気車研究会]]『[[鉄道ピクトリアル]]』増刊 鉄道車両年鑑など。}}

In [12]:
def canonical_vehicle_key(entry: dict) -> tuple[str, str]:
    wiki_base = entry["wiki_title"].split("#", 1)[0]
    return entry["series"], wiki_base


def score_entry(entry: dict) -> int:
    return sum(bool(entry.get(field)) for field in ["type", "subtype", "status", "wiki_title"])


def add_unique(items: list, item):
    if item not in items:
        items.append(item)


raw_series = []
for page_title, (operator_jp, operator_en, wt) in wikitexts.items():
    entries = parse_vehicle_wikitext(wt.splitlines("\n"))
    for e in entries:
        e["operator_page_title"] = page_title
        e["operator_jp"] = operator_jp
        e["operator_en"] = operator_en
    raw_series.extend(entries)

# 同一车型跨 JR 来源页合并；operator/page_title 保留为列表，不再因为重复而丢掉来源。
merged: dict[tuple[str, str], dict] = {}
for e in raw_series:
    key = canonical_vehicle_key(e)

    if key not in merged:
        merged[key] = {
            "series": e["series"],
            "wiki_title": e["wiki_title"],
            "status": e["status"],
            "type": e["type"],
            "subtype": e["subtype"],
            "operator_page_title": [e["operator_page_title"]],
            "operator_jp": [e["operator_jp"]],
            "operator_en": [e["operator_en"]],
            "full_name": e["wiki_title"],
        }
        continue

    current = merged[key]
    if score_entry(e) > score_entry(current):
        for field in ["wiki_title", "status", "type", "subtype"]:
            current[field] = e[field]
        current["full_name"] = e["wiki_title"]

    add_unique(current["operator_page_title"], e["operator_page_title"])
    add_unique(current["operator_jp"], e["operator_jp"])
    add_unique(current["operator_en"], e["operator_en"])

all_series = list(merged.values())
all_series


[{'series': 'E2系',
  'wiki_title': '新幹線E2系電車',
  'status': '現役',
  'type': '新幹線電車',
  'subtype': '営業用',
  'operator_page_title': ['JR東日本の車両形式'],
  'operator_jp': ['JR東日本'],
  'operator_en': ['JR East'],
  'full_name': '新幹線E2系電車'},
 {'series': 'E3系',
  'wiki_title': '新幹線E3系電車',
  'status': '現役',
  'type': '新幹線電車',
  'subtype': '営業用',
  'operator_page_title': ['JR東日本の車両形式'],
  'operator_jp': ['JR東日本'],
  'operator_en': ['JR East'],
  'full_name': '新幹線E3系電車'},
 {'series': 'E5系',
  'wiki_title': '新幹線E5系・H5系電車',
  'status': '現役',
  'type': '新幹線電車',
  'subtype': '営業用',
  'operator_page_title': ['JR東日本の車両形式'],
  'operator_jp': ['JR東日本'],
  'operator_en': ['JR East'],
  'full_name': '新幹線E5系・H5系電車'},
 {'series': 'E6系',
  'wiki_title': '新幹線E6系電車',
  'status': '現役',
  'type': '新幹線電車',
  'subtype': '営業用',
  'operator_page_title': ['JR東日本の車両形式'],
  'operator_jp': ['JR東日本'],
  'operator_en': ['JR East'],
  'full_name': '新幹線E6系電車'},
 {'series': 'E7系',
  'wiki_title': '新幹線E7系・W7系電車',
  'status': '現役',

In [13]:
# 特例覆盖：key=(series, wiki_title 去掉 #anchor)，value=需要覆盖的字段 dict
OVERRIDES: dict[tuple[str, str], dict] = {
    ("E001形", "TRAIN SUITE 四季島"): {"full_name": "JR東日本E001形クルーズトレイン「TRAIN SUITE 四季島」"},
}

for e in all_series:
    override = OVERRIDES.get(canonical_vehicle_key(e))
    if override:
        e.update(override)


In [14]:
all_df = pd.DataFrame(all_series)
all_df.head()


,series,wiki_title,status,type,subtype,operator_page_title,operator_jp,operator_en,full_name
0,E2系,新幹線E2系電車,現役,新幹線電車,営業用,[JR東日本の車両形式],[JR東日本],[JR East],新幹線E2系電車
1,E3系,新幹線E3系電車,現役,新幹線電車,営業用,[JR東日本の車両形式],[JR東日本],[JR East],新幹線E3系電車
2,E5系,新幹線E5系・H5系電車,現役,新幹線電車,営業用,[JR東日本の車両形式],[JR東日本],[JR East],新幹線E5系・H5系電車
3,E6系,新幹線E6系電車,現役,新幹線電車,営業用,[JR東日本の車両形式],[JR東日本],[JR East],新幹線E6系電車
4,E7系,新幹線E7系・W7系電車,現役,新幹線電車,営業用,[JR東日本の車両形式],[JR東日本],[JR East],新幹線E7系・W7系電車


In [15]:
all_df['type'].value_counts()

type
電車                     84
電気機関車                  26
気動車                    22
客車                     20
新幹線電車                  18
貨車                     17
ディーゼル機関車               15
蒸気機関車                   4
電気・ディーゼル両用（EDC方式）車両     1
ハイブリッド機関車               1
Name: count, dtype: int64

In [16]:
all_df['subtype'].value_counts()

subtype
一般形      27
特急形      26
事業用      25
         17
直流用      14
通勤形      14
近郊形      13
旧形営業用    11
営業用      10
急行形       9
交流用       7
旧形事業用     7
交直両用      5
長物車       5
液体式       4
有蓋車       3
車掌車       3
無蓋車       2
操重車       2
電気式       2
ホッパ車      1
検重車       1
Name: count, dtype: int64

## Part.2 数据过滤

滤除对象：
- 货车
- 客车
- 旧型事业车、营业车

In [17]:
#滤除对象：货车，因为多为货列连挂，难以找到单独的车辆照片，一阶段暂时跳过；客车，理由类似，一阶段保留
excluding_types = ["貨車", "客車"]
filtered_df = all_df[~all_df['type'].isin(excluding_types)]
filtered_df['type'].value_counts()
filtered_df['subtype'].value_counts()
filtered_df[filtered_df['subtype'] == "旧形営業用"]

,series,wiki_title,status,type,subtype,operator_page_title,operator_jp,operator_en,full_name
18,クモハ12形,国鉄31系電車,現役,電車,旧形営業用,[JR東日本の車両形式],[JR東日本],[JR East],国鉄31系電車
100,クモハ40形,国鉄40系電車,廃止,電車,旧形営業用,[JR東日本の車両形式],[JR東日本],[JR East],国鉄40系電車


In [18]:
#现在滤除二级，对象为旧式营业车和事业用车
exclduing_subtypes = ["旧形営業用", "旧形事業用"]
final_df = filtered_df[~filtered_df['subtype'].isin(exclduing_subtypes)]
final_df['type'].value_counts()
final_df['subtype'].value_counts()


subtype
一般形     26
事業用     24
特急形     22
        16
直流用     14
通勤形     14
近郊形     13
営業用     10
急行形      8
交流用      7
交直両用     5
液体式      4
電気式      2
Name: count, dtype: int64

In [30]:
# 按 series 去重：同一车型可能同时出现在多个 JR 来源页，保留一行并合并 operator 信息。
def _as_list(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    return [value]


def _extend_unique(items: list, values) -> None:
    for value in _as_list(values):
        if value not in items:
            items.append(value)


def _row_quality(row: pd.Series) -> tuple:
    title = row.get("wiki_title") or ""
    full_name = row.get("full_name") or ""
    return (
        title != row.get("series"),
        len(title),
        len(full_name),
        pd.notna(row.get("subtype")),
    )


duplicate_rows = final_df[final_df.duplicated(subset=["series"], keep=False)]
display(duplicate_rows)

merged_rows = []
for _, group in final_df.groupby("series", sort=False):
    if len(group) == 1:
        merged_rows.append(group.iloc[0].copy())
        continue

    # 主行选信息量更高的标题；operator/page 信息从所有重复行合并。
    base = group.loc[max(group.index, key=lambda idx: _row_quality(group.loc[idx]))].copy()
    for col in ["operator_page_title", "operator_jp", "operator_en"]:
        merged = []
        for value in group[col]:
            _extend_unique(merged, value)
        base[col] = merged

    for col in ["status", "type", "subtype", "wiki_title", "full_name"]:
        if pd.isna(base[col]) or base[col] == "":
            first_valid = group[col].dropna()
            if not first_valid.empty:
                base[col] = first_valid.iloc[0]

    merged_rows.append(base)

final_df = pd.DataFrame(merged_rows).reset_index(drop=True)

print(f"去重前重复行 {len(duplicate_rows)} 条；去重后剩余重复 series {final_df.duplicated(subset=['series']).sum()} 条")
final_df[final_df["series"] == "DD51形"]

,series,wiki_title,status,type,subtype,operator_page_title,operator_jp,operator_en,full_name


去重前重复行 0 条；去重后剩余重复 series 0 条


,series,wiki_title,status,type,subtype,operator_page_title,operator_jp,operator_en,full_name
16,DD51形,国鉄DD51形ディーゼル機関車,現役,ディーゼル機関車,,"[JR東日本の車両形式, JR貨物の車両形式]","[JR東日本, JR貨物]","[JR East, JR Freight]",国鉄DD51形ディーゼル機関車


In [31]:
os.makedirs(os.path.join(PROJECT_ROOT, "data"), exist_ok=True)
export_df = final_df.copy()
for col in ["operator_page_title", "operator_jp", "operator_en"]:
    export_df[col] = export_df[col].apply(lambda v: json.dumps(v, ensure_ascii=False))
export_df.to_csv(os.path.join(PROJECT_ROOT, "data", "jr_east_freight_series.csv"), index=False, encoding="utf-8")
export_df


,series,wiki_title,status,type,subtype,operator_page_title,operator_jp,operator_en,full_name
0,E2系,新幹線E2系電車,現役,新幹線電車,営業用,"[""JR東日本の車両形式""]","[""JR東日本""]","[""JR East""]",新幹線E2系電車
1,E3系,新幹線E3系電車,現役,新幹線電車,営業用,"[""JR東日本の車両形式""]","[""JR東日本""]","[""JR East""]",新幹線E3系電車
2,E5系,新幹線E5系・H5系電車,現役,新幹線電車,営業用,"[""JR東日本の車両形式""]","[""JR東日本""]","[""JR East""]",新幹線E5系・H5系電車
3,E6系,新幹線E6系電車,現役,新幹線電車,営業用,"[""JR東日本の車両形式""]","[""JR東日本""]","[""JR East""]",新幹線E6系電車
4,E7系,新幹線E7系・W7系電車,現役,新幹線電車,営業用,"[""JR東日本の車両形式""]","[""JR東日本""]","[""JR East""]",新幹線E7系・W7系電車
...,...,...,...,...,...,...,...,...,...
159,EF67形,国鉄EF67形電気機関車,廃止車両（貨車以外）,電気機関車,直流用,"[""JR貨物の車両形式""]","[""JR貨物""]","[""JR Freight""]",国鉄EF67形電気機関車
160,EF200形,JR貨物EF200形電気機関車,廃止車両（貨車以外）,電気機関車,直流用,"[""JR貨物の車両形式""]","[""JR貨物""]","[""JR Freight""]",JR貨物EF200形電気機関車
161,ED500形,JR貨物ED500形電気機関車,廃止車両（貨車以外）,電気機関車,交直両用,"[""JR貨物の車両形式""]","[""JR貨物""]","[""JR Freight""]",JR貨物ED500形電気機関車
162,EF500形,JR貨物EF500形電気機関車,廃止車両（貨車以外）,電気機関車,交直両用,"[""JR貨物の車両形式""]","[""JR貨物""]","[""JR Freight""]",JR貨物EF500形電気機関車
